In [1]:
record_name = '082620_355l'

In [2]:
import sys
sys.path.append('../src')
from should_be_stdlib import is_array_lesser
from circuit_postprocess import *
from circuits import (
    circuit_angle_swap,
    circuit_angle_qft_swap,
    circuit_amp_iamp,
    circuit_amp_iamp_qft,
)
from data import *

In [3]:
from itertools import combinations

import pennylane as qml
import pandas as pd
from tqdm.notebook import tqdm

In [4]:
from qbraid import transpile

/home/user/work/quadrigems/.venv/lib/python3.11/site-packages/qbraid/_entrypoints.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
tuning_curves_rescaled = pd.read_csv(datapath(record_name, 'data_tuning-curves_rescaled.csv'), index_col=0)
tuning_curves_rescaled

,3.0,4.2,6.0,8.5,12.0,17.0,24.0,33.9,48.0
0,0.467292,0.305490,0.380194,0.389118,0.320281,0.371346,0.216382,0.237342,0.454148
1,0.358862,0.377713,0.363473,0.298196,0.514289,0.344421,0.270568,0.288043,0.326027
2,0.428871,0.456197,0.353215,0.272013,0.393939,0.269435,0.263589,0.296251,0.408083
3,0.422356,0.435714,0.402926,0.202517,0.426203,0.376766,0.278505,0.176908,0.419698
4,0.366559,0.462955,0.368741,0.297812,0.336580,0.366013,0.282479,0.313760,0.346694
...,...,...,...,...,...,...,...,...,...
747,0.298203,0.302967,0.370425,0.359573,0.319490,0.351042,0.341345,0.469665,0.328883
748,0.349402,0.316119,0.310304,0.290567,0.542874,0.369870,0.217389,0.172724,0.572344
749,0.418599,0.359557,0.349582,0.367598,0.346034,0.329741,0.290123,0.242201,0.438157
750,0.426633,0.364458,0.391670,0.411024,0.381619,0.261847,0.217206,0.253827,0.433309


In [6]:
tuning_curves_resampled = pd.read_csv(datapath(record_name, 'data_tuning-curves_resampled.csv'), index_col=0)
tuning_curves_resampled

,3.0,3.6,4.3,5.2,6.3,7.6,9.1,10.9,13.2,15.8,19.0,22.9,27.6,33.2,39.9,48.0
0,16.021952,12.264374,10.553161,12.029935,13.122600,13.450016,13.010839,11.518352,11.122062,12.595746,11.502252,7.909172,7.048282,7.917080,10.621251,15.571267
1,20.064707,20.752571,21.122397,20.827353,20.051630,17.770461,17.547947,26.211134,27.404850,20.978064,17.578633,15.242605,15.379164,15.998573,16.954879,18.228867
2,21.028196,22.277563,22.195533,19.621125,16.698515,14.074841,13.886597,18.145316,18.363760,14.071747,12.964050,12.832966,13.385413,14.358616,16.401317,20.008920
3,20.785587,21.245353,21.440447,20.860618,19.153457,12.819672,10.760235,18.601808,20.675987,19.282885,17.112897,14.406771,11.495370,8.899217,11.143003,20.654751
4,18.029279,21.411037,22.614818,20.201781,17.622655,15.291523,14.855633,15.996911,17.019773,17.930467,17.045457,14.110422,14.316464,15.329190,16.165761,17.052216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,18.318777,18.182978,18.752275,21.199309,22.812186,22.574898,21.651194,20.058235,19.826921,21.194114,21.519719,20.893221,23.251630,28.491325,27.649828,20.203424
748,16.204639,15.252283,14.632428,14.494233,14.314331,13.749321,14.395782,22.753331,24.111655,18.925143,14.775293,10.859066,8.378430,7.867087,13.051002,26.544237
749,11.726129,10.699232,10.043401,9.818680,9.843671,10.184279,10.243572,9.877369,9.576391,9.360883,8.936871,8.298417,7.565123,6.844920,7.990347,12.274010
750,22.926113,20.658770,19.633473,20.406698,21.190711,21.834027,22.063205,21.249501,19.241030,15.179924,12.950753,11.732424,11.985682,13.379365,16.819925,23.284852


# Transpile circuits

- To transpile a pennylane circuit to qasm2 (for qiskit), run it under a quantum tape, then use the qbraid transpiler
- IonQ transpiles to a dictionary which can also be saved for later

In [7]:
def circuit_to_tape(pl_circuit):
    def tape_machine(*args):
        with qml.tape.QuantumTape() as tape:
            pl_circuit(*args)
        return tape

    return tape_machine

In [8]:
tape = {
    'ang': circuit_to_tape(circuit_angle_swap),
    'ang-qft': circuit_to_tape(circuit_angle_qft_swap),
    'amp': circuit_to_tape(circuit_amp_iamp),
    'amp-qft': circuit_to_tape(circuit_amp_iamp_qft),
}

In [9]:
def small_then_big_array(data1, data2):
    if is_array_lesser(data2, data1):
        return data2, data1
    else:
        return data1, data2

In [10]:
from multiprocessing import Pool, cpu_count

In [11]:
adj_tuning_curves = {
    'ang': tuning_curves_rescaled,
    'ang-qft': tuning_curves_rescaled,
    'amp': tuning_curves_resampled,
    'amp-qft': tuning_curves_resampled
}

In [21]:
def get_fidelity_circuits_sub1(name_a_b):
    name, a, b = name_a_b
    return [
        a,
        b,
        transpile(
            tape[name](
                *small_then_big_array(
                    adj_tuning_curves[name].loc[a].to_numpy(),
                    adj_tuning_curves[name].loc[b].to_numpy()
                )
            ),
            'qasm2'
        )
    ]

def get_fidelity_circuits(name):

    datum = adj_tuning_curves[name]
    pairs = combinations(datum.index, 2)
    pairs_len = len(datum) * (len(datum) - 1) // 2

    with Pool(processes=cpu_count()) as pool:
        # A x B
        ab = list(tqdm(pool.imap(get_fidelity_circuits_sub1, [(name,a,b) for (a,b) in pairs]), total=pairs_len))
        # A x A (should all be 1)
        aa = list(tqdm(pool.imap(get_fidelity_circuits_sub1, [(name,a,a) for a in datum.index]), total=len(datum)))

    return pd.concat([
        pd.DataFrame(aa, columns=['A', 'B', 'qasm2']),
        pd.DataFrame(ab, columns=['A', 'B', 'qasm2'])
    ])


In [22]:
get_fidelity_circuits('ang').to_excel(datapath(record_name, 'circuits_ang.xlsx'))
get_fidelity_circuits('ang-qft').to_excel(datapath(record_name, 'circuits_ang-qft.xlsx'))
get_fidelity_circuits('amp').to_excel(datapath(record_name, 'circuits_amp.xlsx'))
get_fidelity_circuits('amp-qft').to_excel(datapath(record_name, 'circuits_amp-qft.xlsx'))

  0%|          | 0/282376 [00:00<?, ?it/s]

  0%|          | 0/752 [00:00<?, ?it/s]

  0%|          | 0/282376 [00:00<?, ?it/s]

  0%|          | 0/752 [00:00<?, ?it/s]

  0%|          | 0/282376 [00:00<?, ?it/s]

  0%|          | 0/752 [00:00<?, ?it/s]

  0%|          | 0/282376 [00:00<?, ?it/s]

  0%|          | 0/752 [00:00<?, ?it/s]